# 15 — GridSearchCV tuning, applied to the lag-only feature set (builds on notebooks 13-14)

**Why this notebook exists.** Notebook 13 tested individual hourly lags (1h-24h per sensor)
instead of rolling-mean/std features, but tuned with a quick `RandomizedSearchCV` (3 sampled
combinations, 2-fold CV) -- the same shallow search that undersold the rolling-feature model
before notebook 14's exhaustive `GridSearchCV` fixed it (F2 0.594 -> 0.706 for the rolling
feature set). That raises a fair question: was notebook 13's "lag features lose" conclusion
a real property of lag features, or just an artifact of under-tuning them?

**This notebook answers that directly:** same lag-only feature set as notebook 13
(`selected_feature_cols_lagged_all_vehicles.json`, 103 features -- 4 sequential sensors x up to
24 hourly lags each, plus 10 raw current-moment sensors and 4 calendar features), but tuned with
the exact same full, docs-informed `GridSearchCV` grids as notebook 14, not a sampled search.

**Notebooks 13 and 14 are untouched.** This is a comparison, saved under new filenames. Whichever
feature set wins this apples-to-apples tuning comparison is the one that should become
production; that decision happens after this notebook's real results come back, not before.

**Runtime warning, read before running:** same scale as notebook 14 -- 5-fold CV, up to 500
trees, two full grids (RF and XGBoost). Realistically hours, not minutes. Run this the same way
as notebook 14: locally overnight with `caffeinate -i` running alongside it, or on Google Colab.
See the "How to run this" note at the end of this cell.

**How to run this:**
1. `pip install -r requirements.txt` if you haven't already.
2. From the repo root: `nohup jupyter nbconvert --to notebook --execute --inplace notebooks/15_gridsearch_lagged_all_vehicles.ipynb > notebooks/nb15_run.log 2>&1 &`
3. Keep the machine awake (`caffeinate -i` in a separate terminal tab), or run it on Colab per the
   same steps as notebook 14 (upload this notebook, mount Drive, upload
   `all_vehicles_features_lagged.csv` + `selected_feature_cols_lagged_all_vehicles.json` +
   `models/baseline_results_all_vehicles.json` + `models/gridsearch_results_all_vehicles.json` to
   Drive, fix the file paths, Runtime -> Run all).
4. When it's done, send me the executed notebook and I'll read the results out of it.

In [1]:
import json

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix, fbeta_score,
                              f1_score, recall_score, precision_score, make_scorer)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
f2_scorer = make_scorer(fbeta_score, beta=2, pos_label=True)

## 1. Load — the lag-only feature set from notebook 13, same stratified split

103 features: 10 raw current-moment sensors, 4 sequential sensors x up to 24 individual hourly
lags each (96 candidates, 7 dropped by the MI-vs-noise check), and 4 calendar features. No
`roll_mean_*` / `roll_std_*` anywhere in this feature set -- that's the whole point of the
comparison.

In [2]:
df = pd.read_csv("../data/processed/all_vehicles_features_lagged.csv", parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

with open("../data/processed/selected_feature_cols_lagged_all_vehicles.json") as f:
    SELECTED_FEATURE_COLS = json.load(f)

TARGET = "Fault_Within_6h"
df[TARGET] = df[TARGET].astype(bool)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[TARGET], random_state=42)
X_train, y_train = train_df[SELECTED_FEATURE_COLS], train_df[TARGET]
X_test, y_test = test_df[SELECTED_FEATURE_COLS], test_df[TARGET]

print(f"Train: {len(X_train):,} rows, Test: {len(X_test):,} rows, {len(SELECTED_FEATURE_COLS)} features")

Train: 140,064 rows, Test: 35,016 rows, 103 features


## 2. Random Forest — same full grid as notebook 14

Identical grid, identical fix from the coach's original pasted code (`"scaler"` not `"scalar"`,
`random_state` set throughout). The only thing that changed is which 103 features are being fed
in.

In [3]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_param_space = {
    "clf__n_estimators": [100, 200, 300, 400, 500],
    "clf__max_depth": [5, 6, 7, 8],
    "clf__min_samples_leaf": [2, 4, 6],
    "scaler": [StandardScaler(), None],
}

rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(class_weight="balanced", random_state=42)),
])

rf_search = GridSearchCV(
    rf_pipeline,
    rf_param_space,
    cv=skf,
    scoring=f2_scorer,
    n_jobs=-1,
    verbose=2,
)
rf_search.fit(X_train, y_train)

print(f"Best RF params: {rf_search.best_params_}")
print(f"Best RF CV F2 (full training set): {rf_search.best_score_:.6f}")

rf_best = rf_search.best_estimator_
print("GridSearchCV already refit rf_best on the full training set (refit=True is the default).")

Fitting 5 folds for each of 120 candidates, totalling 600 fits


Best RF params: {'clf__max_depth': 8, 'clf__min_samples_leaf': 4, 'clf__n_estimators': 400, 'scaler': None}
Best RF CV F2 (full training set): 0.490449
GridSearchCV already refit rf_best on the full training set (refit=True is the default).


## 3. XGBoost — same full grid as notebook 14

Same `n_estimators` / `max_depth` / `learning_rate` grid pulled from XGBoost's own tuning docs,
same size and shape as notebook 14's, for a fair comparison.

In [4]:
neg, pos = (~y_train).sum(), y_train.sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {neg}/{pos} = {scale_pos_weight:.6f}")

xgb_param_space = {
    "clf__n_estimators": [100, 200, 300, 400, 500],
    "clf__max_depth": [3, 4, 5, 6],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "scaler": [StandardScaler(), None],
}

xgb_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(
        objective="binary:logistic", eval_metric="logloss",
        scale_pos_weight=scale_pos_weight, random_state=42,
    )),
])

xgb_search = GridSearchCV(
    xgb_pipeline,
    xgb_param_space,
    cv=skf,
    scoring=f2_scorer,
    n_jobs=-1,
    verbose=2,
)
xgb_search.fit(X_train, y_train)

print(f"Best XGB params: {xgb_search.best_params_}")
print(f"Best XGB CV F2 (full training set): {xgb_search.best_score_:.6f}")

xgb_best = xgb_search.best_estimator_
print("GridSearchCV already refit xgb_best on the full training set (refit=True is the default).")

scale_pos_weight = 127380/12684 = 10.042573
Fitting 5 folds for each of 120 candidates, totalling 600 fits


[CV] END clf__max_depth=5, clf__min_samples_leaf=2, clf__n_estimators=100, scaler=None; total time=  32.1s
[CV] END clf__max_depth=5, clf__min_samples_leaf=2, clf__n_estimators=200, scaler=StandardScaler(); total time= 1.0min
[CV] END clf__max_depth=5, clf__min_samples_leaf=2, clf__n_estimators=300, scaler=StandardScaler(); total time= 1.5min
[CV] END clf__max_depth=5, clf__min_samples_leaf=2, clf__n_estimators=300, scaler=None; total time= 1.5min
[CV] END clf__max_depth=5, clf__min_samples_leaf=2, clf__n_estimators=400, scaler=None; total time= 1.8min
[CV] END clf__max_depth=5, clf__min_samples_leaf=2, clf__n_estimators=500, scaler=None; total time= 2.1min
[CV] END clf__max_depth=5, clf__min_samples_leaf=4, clf__n_estimators=100, scaler=None; total time=  26.2s
[CV] END clf__max_depth=5, clf__min_samples_leaf=4, clf__n_estimators=200, scaler=StandardScaler(); total time=  52.5s
[CV] END clf__max_depth=5, clf__min_samples_leaf=4, clf__n_estimators=200, scaler=None; total time=  51.3s
[

/Users/harsha/.pyenv/versions/3.11.3/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best XGB params: {'clf__learning_rate': 0.05, 'clf__max_depth': 6, 'clf__n_estimators': 500, 'scaler': None}
Best XGB CV F2 (full training set): 0.534265
GridSearchCV already refit xgb_best on the full training set (refit=True is the default).


## 4. Evaluate on the held-out test set

In [5]:
rf_pred = rf_best.predict(X_test)
xgb_pred = xgb_best.predict(X_test)

for name, pred in [("Random Forest (lagged, GridSearchCV)", rf_pred), ("XGBoost (lagged, GridSearchCV)", xgb_pred)]:
    print("=" * 55)
    print(name.upper())
    print("=" * 55)
    print(classification_report(y_test, pred, labels=[False, True], digits=6, zero_division=0))

RANDOM FOREST (LAGGED, GRIDSEARCHCV)
              precision    recall  f1-score   support

       False   0.976261  0.617303  0.756353     31845
        True   0.180981  0.849259  0.298377      3171

    accuracy                       0.638308     35016
   macro avg   0.578621  0.733281  0.527365     35016
weighted avg   0.904242  0.638308  0.714880     35016

XGBOOST (LAGGED, GRIDSEARCHCV)
              precision    recall  f1-score   support

       False   0.972963  0.775224  0.862910     31845
        True   0.257700  0.783664  0.387857      3171

    accuracy                       0.775988     35016
   macro avg   0.615332  0.779444  0.625384     35016
weighted avg   0.908190  0.775988  0.819890     35016



## 5. The scoreboard that actually answers the question

Quoting the rolling-feature results from both notebook 12 (`RandomizedSearchCV`) and notebook 14
(`GridSearchCV`), and the lag-only result from notebook 13 (`RandomizedSearchCV`), so this
notebook's `GridSearchCV`-tuned lag-only result sits next to all three for a real, honest
comparison: is lag-only competitive with the current production model once tuned equally
thoroughly, not just "did it improve over notebook 13."


In [6]:
with open("../models/baseline_results_all_vehicles.json") as f:
    randomized_rolling = json.load(f)
with open("../models/gridsearch_results_all_vehicles.json") as f:
    gridsearch_rolling = json.load(f)
with open("../models/lagged_results_all_vehicles.json") as f:
    randomized_lagged = json.load(f)

gridsearch_lagged = {}
for name, pred in [("Random Forest (lagged, GridSearchCV)", rf_pred),
                    ("XGBoost (lagged, GridSearchCV)", xgb_pred)]:
    gridsearch_lagged[name] = {
        "recall_fault": recall_score(y_test, pred, pos_label=True, zero_division=0),
        "precision_fault": precision_score(y_test, pred, pos_label=True, zero_division=0),
        "F2_fault": fbeta_score(y_test, pred, beta=2, pos_label=True, zero_division=0),
        "F1_fault_secondary": f1_score(y_test, pred, pos_label=True, zero_division=0),
    }

combined = {
    "XGBoost, rolling features, RandomizedSearchCV (notebook 12)": randomized_rolling["XGBoost (tuned, selected features)"],
    "XGBoost, lagged features, RandomizedSearchCV (notebook 13)": randomized_lagged["XGBoost (lagged features)"],
    "XGBoost, rolling features, GridSearchCV (notebook 14, current production)": gridsearch_rolling["XGBoost (GridSearchCV, full grid)"],
    "XGBoost, lagged features, GridSearchCV (this notebook)": gridsearch_lagged["XGBoost (lagged, GridSearchCV)"],
}
scoreboard = pd.DataFrame(combined).T.round(6)

with open("../models/gridsearch_lagged_results_all_vehicles.json", "w") as f:
    json.dump(gridsearch_lagged, f, indent=2)

scoreboard

,recall_fault,precision_fault,F2_fault,F1_fault_secondary
"XGBoost, rolling features, RandomizedSearchCV (notebook 12)",0.853989,0.268013,0.594172,0.407985
"XGBoost, lagged features, RandomizedSearchCV (notebook 13)",0.823715,0.216422,0.527613,0.342782
"XGBoost, rolling features, GridSearchCV (notebook 14, current production)",0.854620,0.416411,0.706023,0.559975
"XGBoost, lagged features, GridSearchCV (this notebook)",0.783664,0.257700,0.556501,0.387857


In [7]:
production_f2 = gridsearch_rolling["XGBoost (GridSearchCV, full grid)"]["F2_fault"]
lagged_gridsearch_f2 = gridsearch_lagged["XGBoost (lagged, GridSearchCV)"]["F2_fault"]
delta = lagged_gridsearch_f2 - production_f2
direction = "beats" if delta > 0 else ("loses to" if delta < 0 else "ties")

print(f"Current production (rolling features, GridSearchCV): F2 = {production_f2:.6f}")
print(f"Lag-only features, GridSearchCV (this notebook): F2 = {lagged_gridsearch_f2:.6f}")
print(f"Delta: {delta:+.6f} -- lag-only, properly tuned, {direction} the current production model.")

Current production (rolling features, GridSearchCV): F2 = 0.706023
Lag-only features, GridSearchCV (this notebook): F2 = 0.556501
Delta: -0.149522 -- lag-only, properly tuned, loses to the current production model.


## 6. Save the lagged + GridSearchCV models (new filenames, nothing else touched)

In [8]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(rf_best, "../models/random_forest_all_vehicles_lagged_gridsearch.joblib")
joblib.dump(xgb_best, "../models/xgboost_all_vehicles_lagged_gridsearch.joblib")
print("Saved ../models/random_forest_all_vehicles_lagged_gridsearch.joblib and "
      "../models/xgboost_all_vehicles_lagged_gridsearch.joblib")

Saved ../models/random_forest_all_vehicles_lagged_gridsearch.joblib and ../models/xgboost_all_vehicles_lagged_gridsearch.joblib


## 7. What this notebook establishes

A direct test of whether notebook 13's "lag-only features underperform" conclusion was a real
finding or an artifact of insufficient tuning. The result above is quoted honestly whichever way
it goes: if lag-only + `GridSearchCV` beats the current production model, that should become the
new production model (rolling features get a fair, hard-won retirement). If it doesn't, that
confirms the earlier conclusion was correct even after being given a fair, equally-thorough
tuning budget -- and rolling-mean features earn their place in the production model twice over,
not just by default.